# SpectralNP: A Sensor-Agnostic Foundation Model for Spectral Remote Sensing

**Key idea**: A single model that accepts spectral measurements from *any* sensor — regardless of number of bands, spectral resolution, or wavelength positions — and naturally produces higher uncertainty when given less spectral information.

### Architecture at a glance

```
Input: {(λᵢ, FWHMᵢ, Lᵢ)} — N bands from ANY sensor
    ↓
Band Encoder — learnable wavelength encoding + conditioned radiance features
    ↓
Spectral Aggregator — Transformer with RoPE on wavelength + Neural Process dual path
    ↓
z ~ q(z|context) — stochastic latent whose width ∝ 1/N_bands
    ↓
Task Decoders — spectral reconstruction, material ID, atmospheric estimation
    ↓
Output: predictions + calibrated uncertainty
```

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings("ignore")

# Local imports
from spectralnp.model.spectralnp import SpectralNP, SpectralNPConfig
from spectralnp.data.usgs_speclib import load_from_directory
from spectralnp.data.rtm_simulator import simplified_toa_radiance, AtmosphericState, ViewGeometry
from spectralnp.data.random_sensor import sample_virtual_sensor, apply_sensor, add_sensor_noise
from spectralnp.data.sensor_definitions import LANDSAT8_OLI, SENTINEL2_MSI, AVIRIS_NG, SENSORS
from spectralnp.inference.predict import SpectralNPPredictor

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.dpi": 120,
})

print("✓ All imports loaded")


---
## 1. Load Trained Model

Load a pre-trained SpectralNP checkpoint. The model has **~5M parameters** and was trained on synthetic spectra (minerals, vegetation, soil, water, man-made materials) passed through atmospheric radiative transfer simulation with random sensor augmentation.

In [ ]:
# Load the trained model
ckpt = torch.load("demo_model.pt", map_location="cpu", weights_only=False)
model = SpectralNP.from_checkpoint(ckpt)
predictor = SpectralNPPredictor(model)
print(f"Loaded model: {sum(p.numel() for p in model.parameters()):,} parameters")


---
## 2. Training Data Pipeline

The model is trained on **simulated at-sensor radiance**. We take surface reflectance spectra, pass them through atmospheric radiative transfer, then observe through randomly-generated virtual sensors.

In [ ]:
# Load real USGS spectra (same library the model was trained on)
from spectralnp.data.usgs_speclib import load_from_directory

speclib = load_from_directory("/Users/eric/repo/atmgen/USGS_ASCIIdata/ASCIIdata_splib07a")
speclib = speclib.filter_wavelength_range(380, 2400)
wl_dense = np.arange(380.0, 2501.0, 5.0)

# Build an index → category lookup matching the order the model was trained on.
category_by_idx = [s.category for s in speclib.spectra]
all_categories = sorted(set(category_by_idx))
print(f"Loaded {len(speclib)} USGS spectra across {len(all_categories)} categories")
for cat in all_categories:
    n = sum(1 for c in category_by_idx if c == cat)
    print(f"  {cat:12s}  {n:4d} spectra")

# Plot a few examples from each major category
plot_cats = ["minerals", "vegetation", "soils", "artificial", "organics"]
colors = ["#e74c3c", "#27ae60", "#8B4513", "#7f8c8d", "#9b59b6"]

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=True)
for ax, cat, col in zip(axes, plot_cats, colors):
    subset = speclib.filter_category(cat)
    for s in subset.spectra[:8]:
        refl = np.clip(np.nan_to_num(s.resample(wl_dense)), 0, 1)
        ax.plot(wl_dense, refl, color=col, alpha=0.5, linewidth=0.8)
    ax.set_title(f"{cat.capitalize()} (n={len(subset)})", fontweight="bold")
    ax.set_xlabel("Wavelength (nm)")
    ax.set_xlim(380, 2500)

axes[0].set_ylabel("Reflectance")
fig.suptitle("USGS Spectral Library v7 — Real Surface Reflectance",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Show the RTM pipeline: reflectance → at-sensor radiance under different atmospheres
veg_spec = speclib.filter_category("vegetation").spectra[3]
refl = veg_spec.resample(wl_dense)
refl = np.clip(np.nan_to_num(refl), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Surface reflectance
axes[0].plot(wl_dense, refl, color="#27ae60", linewidth=1.5)
axes[0].set_title("Surface Reflectance", fontweight="bold")
axes[0].set_ylabel("Reflectance")
axes[0].set_xlabel("Wavelength (nm)")

# Panel 2: At-sensor radiance under different atmospheres
atmos_configs = [
    ("Clear (AOD=0.05)", AtmosphericState(aod_550=0.05, water_vapour=0.5)),
    ("Moderate (AOD=0.3)", AtmosphericState(aod_550=0.3, water_vapour=2.0)),
    ("Hazy (AOD=0.8)", AtmosphericState(aod_550=0.8, water_vapour=4.0)),
]
for label, atmos in atmos_configs:
    result = simplified_toa_radiance(refl, wl_dense, atmos=atmos)
    axes[1].plot(wl_dense, result.toa_radiance, linewidth=1.2, label=label)
axes[1].set_title("At-Sensor Radiance (RTM)", fontweight="bold")
axes[1].set_ylabel("Radiance (W/m²/sr/μm)")
axes[1].set_xlabel("Wavelength (nm)")
axes[1].legend(fontsize=9)

# Panel 3: What different sensors see (same spectrum)
result = simplified_toa_radiance(refl, wl_dense, atmos=AtmosphericState(aod_550=0.15))
for name, sensor, color in [("Landsat-8 (7 bands)", LANDSAT8_OLI, "#e74c3c"),
                              ("Sentinel-2 (13 bands)", SENTINEL2_MSI, "#3498db"),
                              ("AVIRIS-NG (424 bands)", AVIRIS_NG, "#27ae60")]:
    bands = sensor.convolve(wl_dense, result.toa_radiance)
    axes[2].scatter(sensor.center_wavelength_nm, bands, s=12, label=name, color=color, zorder=3)
axes[2].plot(wl_dense, result.toa_radiance, "k-", alpha=0.2, linewidth=0.8, label="True spectrum")
axes[2].set_title("Sensor Observations", fontweight="bold")
axes[2].set_ylabel("Radiance (W/m²/sr/μm)")
axes[2].set_xlabel("Wavelength (nm)")
axes[2].legend(fontsize=8)

fig.suptitle("Training Data Pipeline: Reflectance → RTM → Sensor Sampling", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Core Capability: Sensor-Agnostic Inference

The **same model** processes data from any sensor. We feed it Landsat-8 (7 bands), Sentinel-2 (13 bands), and AVIRIS-NG (424 bands) — no retraining, no sensor-specific heads.

In [ ]:
# Create a test case: mineral spectrum through moderate atmosphere
mineral_spec = speclib.filter_category("minerals").spectra[2]
refl_mineral = np.clip(np.nan_to_num(mineral_spec.resample(wl_dense)), 0, 1)
atmos_true = AtmosphericState(aod_550=0.2, water_vapour=1.8, ozone_du=320, visibility_km=30)
geom = ViewGeometry(solar_zenith_deg=35)
rtm_result = simplified_toa_radiance(refl_mineral, wl_dense, atmos=atmos_true, geometry=geom)
true_radiance = rtm_result.toa_radiance

query_wl = wl_dense

# Run prediction from each sensor
sensors_to_test = [
    ("Landsat-8\n(7 bands)", LANDSAT8_OLI, "#e74c3c"),
    ("Sentinel-2\n(13 bands)", SENTINEL2_MSI, "#3498db"),
    ("AVIRIS-NG\n(424 bands)", AVIRIS_NG, "#27ae60"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, (name, sensor, color) in zip(axes, sensors_to_test):
    # Simulate what this sensor measures
    band_radiance = sensor.convolve(wl_dense, true_radiance)
    band_radiance = add_sensor_noise(band_radiance, np.random.default_rng(42), snr_range=(200, 200))

    # Predict with SpectralNP
    pred = predictor.predict(
        wavelength_nm=sensor.center_wavelength_nm,
        fwhm_nm=sensor.fwhm_nm,
        radiance=band_radiance,
        query_wavelength_nm=query_wl,
        n_samples=32,
    )

    # Plot
    ax.plot(wl_dense, true_radiance, "k-", linewidth=1.0, alpha=0.4, label="True spectrum")
    ax.plot(query_wl, pred.spectral_mean, color=color, linewidth=1.5, label="Predicted")
    ax.fill_between(query_wl,
                     pred.spectral_mean - 2 * pred.spectral_std,
                     pred.spectral_mean + 2 * pred.spectral_std,
                     color=color, alpha=0.2, label="±2σ uncertainty")
    ax.scatter(sensor.center_wavelength_nm, band_radiance, s=15, color="black",
               zorder=5, label=f"Input ({sensor.n_bands} bands)")
    ax.set_title(name, fontweight="bold", fontsize=12)
    ax.set_xlabel("Wavelength (nm)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(380, 2500)

axes[0].set_ylabel("Radiance (W/m²/sr/μm)")
fig.suptitle("Same Model, Different Sensors — Spectral Reconstruction", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Key Property: Uncertainty Scales with Information Content

The **Neural Process latent variable** `z` has a posterior whose width is inversely proportional to the number of observed bands. This is not a bolted-on heuristic — it's a mathematical property of the architecture.

Below we feed the **same spectrum** through virtual sensors with 3, 7, 13, 30, 100, and 200 bands and measure the resulting uncertainty.

In [ ]:
# Sweep band count and measure uncertainty
band_counts = [3, 5, 7, 13, 30, 50, 100, 200]
mean_spectral_stds = []
mean_posterior_sigmas = []
rng = np.random.default_rng(123)

# Use a vegetation spectrum for this test
veg_refl = np.clip(np.nan_to_num(speclib.filter_category("vegetation").spectra[1].resample(wl_dense)), 0, 1)
veg_result = simplified_toa_radiance(veg_refl, wl_dense, atmos=AtmosphericState(aod_550=0.15))
veg_radiance = veg_result.toa_radiance

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, n_bands in enumerate(band_counts):
    ax = axes[i // 4, i % 4]

    # Create a virtual sensor with exactly n_bands
    sensor = sample_virtual_sensor(rng, n_bands_range=(n_bands, n_bands), strategy="regular")
    band_rad = apply_sensor(sensor, wl_dense, veg_radiance).astype(np.float32)
    band_rad = add_sensor_noise(band_rad, rng, snr_range=(150, 150)).astype(np.float32)

    # Predict
    pred = predictor.predict(
        wavelength_nm=sensor.center_wavelength_nm,
        fwhm_nm=sensor.fwhm_nm,
        radiance=band_rad,
        query_wavelength_nm=query_wl,
        n_samples=32,
    )
    mean_spectral_stds.append(pred.spectral_std.mean())

    # Also get latent posterior width
    with torch.no_grad():
        wl_t = torch.tensor(sensor.center_wavelength_nm).unsqueeze(0)
        fw_t = torch.tensor(sensor.fwhm_nm).unsqueeze(0)
        rad_t = torch.tensor(band_rad).unsqueeze(0)
        # encode() returns (r, z_atm_mu, z_atm_log_sigma, z_surf_mu, z_surf_log_sigma, z_mu, z_log_sigma)
        _, _, _, _, _, z_mu, z_logsig = model.encode(wl_t, fw_t, rad_t)
        mean_posterior_sigmas.append(z_logsig.exp().mean().item())

    # Plot
    ax.plot(wl_dense, veg_radiance, "k-", alpha=0.3, linewidth=0.8)
    ax.plot(query_wl, pred.spectral_mean, "#2ecc71", linewidth=1.2)
    ax.fill_between(query_wl,
                     pred.spectral_mean - 2 * pred.spectral_std,
                     pred.spectral_mean + 2 * pred.spectral_std,
                     color="#2ecc71", alpha=0.25)
    ax.scatter(sensor.center_wavelength_nm, band_rad, s=8, color="black", zorder=5)
    ax.set_title(f"{n_bands} bands", fontweight="bold")
    ax.set_xlim(380, 2500)
    ax.set_ylim(bottom=0)
    if i % 4 == 0:
        ax.set_ylabel("Radiance")

fig.suptitle("Uncertainty vs. Number of Input Bands (same underlying spectrum)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Quantitative: plot uncertainty metrics vs band count
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(band_counts, mean_spectral_stds, "o-", color="#e74c3c", linewidth=2, markersize=8)
ax1.set_xlabel("Number of Input Bands", fontsize=12)
ax1.set_ylabel("Mean Spectral Uncertainty (σ)", fontsize=12)
ax1.set_title("Predictive Uncertainty vs. Band Count", fontweight="bold")
ax1.set_xscale("log")

ax2.plot(band_counts, mean_posterior_sigmas, "s-", color="#8e44ad", linewidth=2, markersize=8)
ax2.set_xlabel("Number of Input Bands", fontsize=12)
ax2.set_ylabel("Mean Latent σ(z)", fontsize=12)
ax2.set_title("Latent Posterior Width vs. Band Count", fontweight="bold")
ax2.set_xscale("log")

fig.suptitle("More bands → Less uncertainty (by construction)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Summary: Uncertainty ratio (3 bands / 200 bands)")
print(f"  Spectral uncertainty: {mean_spectral_stds[0] / mean_spectral_stds[-1]:.1f}x")
print(f"  Latent posterior σ:   {mean_posterior_sigmas[0] / mean_posterior_sigmas[-1]:.1f}x")

---
## 5. Material Identification Across Sensors

The model classifies surface materials from any sensor's input. With more spectral information, the classification entropy decreases (higher confidence).

In [ ]:
# Material classification from different sensors (real USGS spectra)
test_materials = [
    ("Mineral", speclib.filter_category("minerals").spectra[5], "#e74c3c"),
    ("Vegetation", speclib.filter_category("vegetation").spectra[2], "#27ae60"),
    ("Soil", speclib.filter_category("soils").spectra[3], "#8B4513"),
    ("Artificial", speclib.filter_category("artificial").spectra[1], "#7f8c8d"),
]

# Use the model's own category names if available (from checkpoint).
# This handles models that output per-category probs (n_classes=7) directly.
n_model_classes = model.cfg.n_material_classes
model_category_names = ckpt.get("category_names")

fig, axes = plt.subplots(len(test_materials), 3, figsize=(16, 3.2 * len(test_materials)))

for row, (mat_name, spec, color) in enumerate(test_materials):
    refl = np.clip(np.nan_to_num(spec.resample(wl_dense), nan=0.04), 0, 1)
    rtm = simplified_toa_radiance(refl, wl_dense, atmos=AtmosphericState(aod_550=0.2))

    for col, (sensor_name, sensor) in enumerate([
        ("Landsat-8", LANDSAT8_OLI),
        ("Sentinel-2", SENTINEL2_MSI),
        ("AVIRIS-NG", AVIRIS_NG),
    ]):
        ax = axes[row, col]
        band_rad = sensor.convolve(wl_dense, rtm.toa_radiance).astype(np.float32)
        pred = predictor.predict(
            wavelength_nm=sensor.center_wavelength_nm,
            fwhm_nm=sensor.fwhm_nm,
            radiance=band_rad,
            n_samples=16,
        )
        if model_category_names is not None and len(pred.material_probs) == n_model_classes:
            cat_probs = {n: float(p) for n, p in zip(model_category_names, pred.material_probs)}
        else:
            cat_probs = {}
            for idx, p in enumerate(pred.material_probs):
                c = category_by_idx[idx]
                cat_probs[c] = cat_probs.get(c, 0.0) + float(p)
        sorted_cats = sorted(cat_probs.items(), key=lambda x: -x[1])[:5]
        labels = [c for c, _ in sorted_cats]
        probs = [p for _, p in sorted_cats]
        ax.barh(range(len(labels)), probs, color=color, alpha=0.7)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        if row == 0:
            ax.set_title(f"{sensor_name}\n({sensor.n_bands} bands)", fontweight="bold")
        if col == 0:
            ax.set_ylabel(f"{mat_name}", fontweight="bold", fontsize=11)
        if row == len(test_materials) - 1:
            ax.set_xlabel("Category probability")
        ax.text(0.95, 0.95, f"H={pred.material_entropy:.2f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

fig.suptitle("Material Classification — Same Model Across Sensors\n"
             "(model outputs per-USGS-category probabilities)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 6. Atmospheric Parameter Estimation with Evidential Uncertainty

The atmospheric decoder estimates aerosol optical depth (AOD), water vapour, ozone, and visibility directly from at-sensor radiance — with separate **aleatoric** (data noise) and **epistemic** (model uncertainty) estimates via the Normal-Inverse-Gamma parameterisation.

In [ ]:
# Atmospheric parameter estimation across varying conditions
from spectralnp.model.evidential import nig_uncertainty

# Generate a grid of test atmospheres
aod_range = np.linspace(0.05, 0.8, 8)
wv_range = np.linspace(0.3, 4.5, 8)

# Use AVIRIS-NG for best spectral coverage
sensor = AVIRIS_NG
soil_spec = speclib.filter_category("soils").spectra[4]
soil_refl = np.clip(np.nan_to_num(soil_spec.resample(wl_dense)), 0, 1)

# Sweep AOD
aod_pred, aod_true, aod_std = [], [], []
for aod in aod_range:
    atmos = AtmosphericState(aod_550=aod, water_vapour=2.0)
    rtm = simplified_toa_radiance(soil_refl, wl_dense, atmos=atmos)
    band_rad = sensor.convolve(wl_dense, rtm.toa_radiance).astype(np.float32)

    pred = predictor.predict(
        wavelength_nm=sensor.center_wavelength_nm,
        fwhm_nm=sensor.fwhm_nm,
        radiance=band_rad,
        n_samples=32,
    )
    aod_true.append(aod)
    aod_pred.append(pred.atmos_mean[0])  # AOD is first param
    aod_std.append(pred.atmos_std[0])

# Sweep water vapour
wv_pred, wv_true, wv_std = [], [], []
for wv in wv_range:
    atmos = AtmosphericState(aod_550=0.2, water_vapour=wv)
    rtm = simplified_toa_radiance(soil_refl, wl_dense, atmos=atmos)
    band_rad = sensor.convolve(wl_dense, rtm.toa_radiance).astype(np.float32)

    pred = predictor.predict(
        wavelength_nm=sensor.center_wavelength_nm,
        fwhm_nm=sensor.fwhm_nm,
        radiance=band_rad,
        n_samples=32,
    )
    wv_true.append(wv)
    wv_pred.append(pred.atmos_mean[1])  # water vapour is second param
    wv_std.append(pred.atmos_std[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AOD
ax = axes[0]
ax.errorbar(aod_true, aod_pred, yerr=np.array(aod_std) * 2, fmt="o-",
            color="#e74c3c", capsize=4, linewidth=1.5, markersize=6, label="Predicted ±2σ")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="1:1 line")
ax.set_xlabel("True AOD (550nm)", fontsize=12)
ax.set_ylabel("Predicted AOD", fontsize=12)
ax.set_title("Aerosol Optical Depth", fontweight="bold")
ax.legend()

# Water vapour
ax = axes[1]
ax.errorbar(wv_true, wv_pred, yerr=np.array(wv_std) * 2, fmt="s-",
            color="#3498db", capsize=4, linewidth=1.5, markersize=6, label="Predicted ±2σ")
ax.plot([0, 5], [0, 5], "k--", alpha=0.3, label="1:1 line")
ax.set_xlabel("True Water Vapour (g/cm²)", fontsize=12)
ax.set_ylabel("Predicted Water Vapour", fontsize=12)
ax.set_title("Columnar Water Vapour", fontweight="bold")
ax.legend()

fig.suptitle("Atmospheric Parameter Estimation (AVIRIS-NG input, evidential uncertainty)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Mixed-Sensor Batch Inference

A unique capability: process observations from **different sensors in the same batch**. Each pixel can come from a different instrument.

In [ ]:
# Mixed batch: 4 different sensors, 4 different materials, one forward pass
spectra_for_batch = [
    speclib.filter_category("vegetation").spectra[0],
    speclib.filter_category("minerals").spectra[1],
    speclib.filter_category("artificial").spectra[0],
    speclib.filter_category("soils").spectra[2],
]
sensors_for_batch = [LANDSAT8_OLI, SENTINEL2_MSI, AVIRIS_NG, LANDSAT8_OLI]
sensor_names = ["Landsat-8 (vegetation)", "Sentinel-2 (mineral)", "AVIRIS-NG (artificial)", "Landsat-8 (soil)"]

wl_list, fwhm_list, rad_list = [], [], []
for spec, sensor in zip(spectra_for_batch, sensors_for_batch):
    refl = np.clip(np.nan_to_num(spec.resample(wl_dense)), 0, 1)
    rtm = simplified_toa_radiance(refl, wl_dense)
    band_rad = sensor.convolve(wl_dense, rtm.toa_radiance).astype(np.float32)
    wl_list.append(sensor.center_wavelength_nm.astype(np.float32))
    fwhm_list.append(sensor.fwhm_nm.astype(np.float32))
    rad_list.append(band_rad)

# Single batch call with mixed sensors!
batch_pred = predictor.predict_batch(
    wavelength_nm=wl_list,
    fwhm_nm=fwhm_list,
    radiance=rad_list,
    query_wavelength_nm=query_wl,
    n_samples=16,
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
colors = ["#27ae60", "#e74c3c", "#7f8c8d", "#8B4513"]

for i, (ax, name, color) in enumerate(zip(axes, sensor_names, colors)):
    ax.plot(query_wl, batch_pred.spectral_mean[i], color=color, linewidth=1.5)
    ax.fill_between(query_wl,
                     batch_pred.spectral_mean[i] - 2 * batch_pred.spectral_std[i],
                     batch_pred.spectral_mean[i] + 2 * batch_pred.spectral_std[i],
                     color=color, alpha=0.2)
    ax.scatter(wl_list[i], rad_list[i], s=10, color="black", zorder=5)
    ax.set_title(name, fontweight="bold", fontsize=10)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_xlim(380, 2500)

axes[0].set_ylabel("Radiance")
fig.suptitle("Mixed-Sensor Batch — Different instruments in one forward pass",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

n_bands_per = [len(w) for w in wl_list]
print(f"Batch processed: {n_bands_per} bands per observation (padded to {max(n_bands_per)})")


---
## 8. Architecture Deep Dive: Why It Works

The architecture combines three ideas that each solve a different part of the problem:

In [ ]:
# Visualise the learned spectral wavelength encoding
from spectralnp.model.band_encoder import SpectralPositionalEncoding

# Extract learned frequencies from the model
spe = model.encoder.spectral_pos
with torch.no_grad():
    learned_freqs = spe.log_freqs.exp().cpu().numpy()

    # Compute encoding for a sweep of wavelengths
    wl_sweep = torch.linspace(380, 2500, 500)
    fwhm_sweep = torch.full_like(wl_sweep, 10.0)
    encoding = spe(wl_sweep, fwhm_sweep).numpy()  # (500, d_model)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# Panel 1: Learned frequency distribution
ax = axes[0]
ax.bar(range(len(learned_freqs)), np.sort(learned_freqs), color="#8e44ad", alpha=0.7)
ax.set_xlabel("Frequency index (sorted)")
ax.set_ylabel("Frequency (rad/nm)")
ax.set_title("Learned Spectral Frequencies", fontweight="bold")
ax.text(0.95, 0.95, f"n={len(learned_freqs)} frequencies\nspan: {learned_freqs.min():.4f}–{learned_freqs.max():.2f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Panel 2: Encoding similarity matrix (do nearby wavelengths encode similarly?)
ax = axes[1]
# Subsample for visibility
idx = np.linspace(0, 499, 50).astype(int)
sub_enc = encoding[idx]
sim = sub_enc @ sub_enc.T
sim_norm = sim / (np.linalg.norm(sub_enc, axis=1, keepdims=True) @ np.linalg.norm(sub_enc, axis=1, keepdims=True).T + 1e-8)
wl_labels = wl_sweep.numpy()[idx]
im = ax.imshow(sim_norm, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
               extent=[wl_labels[0], wl_labels[-1], wl_labels[-1], wl_labels[0]])
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Wavelength (nm)")
ax.set_title("Encoding Cosine Similarity", fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 3: First few encoding dimensions across wavelength
ax = axes[2]
for dim in range(min(8, encoding.shape[1])):
    ax.plot(wl_sweep.numpy(), encoding[:, dim], alpha=0.6, linewidth=0.8)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Encoding value")
ax.set_title("Spectral Encoding Dimensions", fontweight="bold")
ax.text(0.95, 0.95, "First 8 dims shown",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle("Learned Spectral Positional Encoding", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Summary

| Feature | How it works |
|---|---|
| **Sensor-agnostic** | Each band is a token `(λ, FWHM, L)` — any sensor configuration accepted |
| **Variable band count** | Transformer handles variable-length sequences; latent bottleneck gives fixed-size output |
| **Uncertainty ∝ 1/bands** | Neural Process posterior widens with fewer context points — by construction |
| **At-sensor radiance** | Trained on RTM-simulated TOA radiance; atmospheric effects are learned |
| **Continuous output** | DeepONet-style decoder queries any wavelength — not tied to a grid |
| **Multi-task** | Shared backbone for spectral reconstruction, material ID, and atmospheric estimation |
| **Evidential uncertainty** | NIG output decomposes aleatoric vs epistemic uncertainty |

### Next steps
- **Scale up**: Train on full USGS Spectral Library v7 (~1300 spectra) with PyARTS full RTM
- **Spatial extension**: Add spatial transformer for image-level predictions
- **Real sensor fine-tuning**: Adapt to Landsat/Sentinel-2/AVIRIS with real ground truth
- **Benchmark**: Compare against sensor-specific models on standard RS benchmarks